In [ ]:
from __future__ import annotations


import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASS_NAMES = ["streaky", "transition", "spotty"]
MEMORY_MAX_NORM = 10.0

In [ ]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self, in_channels: int = 3, dr: float = 0.3) -> None:
        super().__init__()
        self.pad1 = nn.ZeroPad2d((2, 2, 0, 0))
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(1, 3))
        self.bn1 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d((1, 2))
        self.pad2 = nn.ZeroPad2d((2, 2, 0, 0))
        self.conv2 = nn.Conv2d(64, 128, kernel_size=(2, 3))
        self.bn2 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d((1, 2))
        self.pad3 = nn.ZeroPad2d((2, 2, 0, 0))
        self.conv3 = nn.Conv2d(128, 256, kernel_size=(2, 3))
        self.bn3 = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d((1, 2))
        self.pad4 = nn.ZeroPad2d((2, 2, 0, 0))
        self.conv4 = nn.Conv2d(256, 256, kernel_size=(1, 3))
        self.bn4 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d((1, 2))
        self.dropout = nn.Dropout(dr)

    def forward(self, x):
        x = self.pad1(x); x = F.relu(self.bn1(self.conv1(x))); x = self.dropout(self.pool1(x))
        x = self.pad2(x); x = F.relu(self.bn2(self.conv2(x))); x = self.dropout(self.pool2(x))
        x = self.pad3(x); x = F.relu(self.bn3(self.conv3(x))); x = self.dropout(self.pool3(x))
        x = self.pad4(x); x = F.relu(self.bn4(self.conv4(x))); x = self.dropout(self.pool4(x))
        return x


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int = 64, num_heads: int = 4, mlp_ratio: int = 2,
                 dropout: float = 0.1) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * mlp_ratio, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x1 = self.norm1(x)
        attn_out, _ = self.attn(x1, x1, x1)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class CNNTransformer(nn.Module):
    def __init__(self, num_classes: int = 3, dr: float = 0.3, projection_dim: int = 64,
                 num_heads: int = 4, transformer_layers: int = 8,
                 input_shape=(3, 32, 64)) -> None:
        super().__init__()
        self.cnn = CNNFeatureExtractor(in_channels=input_shape[0], dr=dr)
        self.projection_dim = projection_dim
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            cnn_out = self.cnn(dummy)
            _, C, H, W = cnn_out.shape
            num_patches = H * W
            flatten_dim = num_patches * projection_dim
        self.patch_proj = nn.Linear(C, projection_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, projection_dim) * 0.02)
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(projection_dim, num_heads, mlp_ratio=2, dropout=0.1)
            for _ in range(transformer_layers)
        ])
        self.norm = nn.LayerNorm(projection_dim, eps=1e-6)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(flatten_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        features = self.cnn(x)
        B, C, H, W = features.shape
        features = features.view(B, C, H * W).transpose(1, 2)
        features = self.patch_proj(features) + self.pos_embed
        for block in self.transformer_blocks:
            features = block(features)
        features = self.norm(features)
        features = self.dropout(features)
        features = features.flatten(1)
        return self.classifier(features)


In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, kernel_size: int) -> None:
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim,
                              kernel_size, padding=kernel_size // 2)

    def forward(self, x, hidden_state):
        h, c = hidden_state
        gates = self.conv(torch.cat([x, h], dim=1))
        i, f, o, g = torch.split(gates, self.hidden_dim, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new


class SelfAttentionMemory(nn.Module):
    def __init__(self, channels: int, reduction: int = 4, spatial_reduction: int = 8,
                 memory_max_norm: float = MEMORY_MAX_NORM) -> None:
        super().__init__()
        self.channels = channels
        self.d_k = channels // reduction
        self.spatial_reduction = spatial_reduction
        self.memory_max_norm = memory_max_norm
        self.W_hq = nn.Conv2d(channels, self.d_k, 1)
        self.W_hk = nn.Conv2d(channels, self.d_k, 1)
        self.W_hv = nn.Conv2d(channels, channels, 1)
        self.W_mk = nn.Conv2d(channels, self.d_k, 1)
        self.W_mv = nn.Conv2d(channels, channels, 1)
        self.W_z = nn.Conv2d(channels * 2, channels, 1)
        self.attn_dropout = nn.Dropout2d(0.1)
        self.W_mg = nn.Conv2d(channels, channels, 1)
        self.W_mo = nn.Conv2d(channels, channels, 1)
        self.W_mi = nn.Conv2d(channels, channels, 1)
        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, h_t, m_prev):
        B, C, H, W = h_t.shape
        sr = self.spatial_reduction
        h_small = F.adaptive_avg_pool2d(h_t, (H // sr, W // sr))
        m_small = F.adaptive_avg_pool2d(m_prev, (H // sr, W // sr))
        _, _, hs, ws = h_small.shape
        sp = hs * ws
        scale = np.sqrt(self.d_k)
        Q_h = self.W_hq(h_small).view(B, self.d_k, sp).permute(0, 2, 1)
        K_h = self.W_hk(h_small).view(B, self.d_k, sp)
        K_m = self.W_mk(m_small).view(B, self.d_k, sp)
        V_h = self.W_hv(h_small).view(B, C, sp)
        V_m = self.W_mv(m_small).view(B, C, sp)
        A_h = F.softmax(torch.bmm(Q_h, K_h) / scale, dim=-1)
        A_m = F.softmax(torch.bmm(Q_h, K_m) / scale, dim=-1)
        Z_h = torch.bmm(V_h, A_h.permute(0, 2, 1)).view(B, C, hs, ws)
        Z_m = torch.bmm(V_m, A_m.permute(0, 2, 1)).view(B, C, hs, ws)
        Z_h = F.interpolate(Z_h, size=(H, W), mode='bilinear', align_corners=False)
        Z_m = F.interpolate(Z_m, size=(H, W), mode='bilinear', align_corners=False)
        Z = self.W_z(torch.cat([Z_h, Z_m], dim=1))
        Z = self.attn_dropout(Z)
        output_gate = torch.sigmoid(self.W_mo(Z))
        h_t_hat = output_gate * torch.tanh(self.W_mg(Z))
        input_gate = torch.sigmoid(self.W_mi(Z))
        m_t = (1 - input_gate) * m_prev + input_gate * h_t_hat
        m_t_norm = torch.norm(m_t, dim=1, keepdim=True).clamp(min=1e-8)
        m_t = torch.where(m_t_norm > self.memory_max_norm,
                          m_t * self.memory_max_norm / m_t_norm, m_t)
        return h_t_hat, m_t


class MSAMConvLSTM(nn.Module):
    def __init__(self, input_channels: int = 1, hidden_dims=None,
                 kernel_size: int = 3, num_layers: int = 3, attention_reduction: int = 4,
                 dropout_rates=None) -> None:
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [64, 128, 128]
        if dropout_rates is None:
            dropout_rates = [0.1, 0.1, 0.2]
        self.num_layers = num_layers
        self.hidden_dims = hidden_dims
        self.encoder_cells = nn.ModuleList()
        self.encoder_sam = nn.ModuleList()
        self.encoder_bns = nn.ModuleList()
        self.encoder_drops = nn.ModuleList()
        self.decoder_cells = nn.ModuleList()
        self.decoder_sam = nn.ModuleList()
        self.decoder_bns = nn.ModuleList()
        self.decoder_drops = nn.ModuleList()
        for i in range(num_layers):
            inp = input_channels if i == 0 else hidden_dims[i - 1]
            self.encoder_cells.append(ConvLSTMCell(inp, hidden_dims[i], kernel_size))
            self.encoder_sam.append(SelfAttentionMemory(hidden_dims[i], attention_reduction))
            self.encoder_bns.append(nn.BatchNorm2d(hidden_dims[i]))
            self.encoder_drops.append(nn.Dropout2d(dropout_rates[i]))
            self.decoder_cells.append(ConvLSTMCell(inp, hidden_dims[i], kernel_size))
            self.decoder_sam.append(SelfAttentionMemory(hidden_dims[i], attention_reduction))
            self.decoder_bns.append(nn.BatchNorm2d(hidden_dims[i]))
            self.decoder_drops.append(nn.Dropout2d(dropout_rates[i]))
        self.output_conv = nn.Conv2d(hidden_dims[-1], input_channels, 1)

    def forward(self, x: torch.Tensor, future_seq: int = 5) -> torch.Tensor:
        B, T, C, H, W = x.shape
        states = [(torch.zeros(B, d, H, W, device=x.device),
                   torch.zeros(B, d, H, W, device=x.device))
                  for d in self.hidden_dims]
        memories = [torch.zeros(B, d, H, W, device=x.device) for d in self.hidden_dims]
        for t in range(T):
            inp = x[:, t]
            h_out = None
            for i in range(self.num_layers):
                h, c = self.encoder_cells[i](inp if i == 0 else h_out, states[i])
                h_out, m_new = self.encoder_sam[i](h, memories[i])
                h_out = self.encoder_drops[i](self.encoder_bns[i](h_out))
                states[i] = (h_out, c)
                memories[i] = m_new
        dec_states = [(h.clone(), c.clone()) for h, c in states]
        dec_memories = [m.clone() for m in memories]
        dec_input = x[:, -1]
        outputs = []
        for t in range(future_seq):
            h_out = None
            for i in range(self.num_layers):
                h, c = self.decoder_cells[i](dec_input if i == 0 else h_out, dec_states[i])
                h_out, m_new = self.decoder_sam[i](h, dec_memories[i])
                h_out = self.decoder_drops[i](self.decoder_bns[i](h_out))
                dec_states[i] = (h_out, c)
                dec_memories[i] = m_new
            out = torch.sigmoid(self.output_conv(h_out))
            outputs.append(out)
            dec_input = out
        return torch.stack(outputs, dim=1)


In [ ]:
def _load_state(model: nn.Module, ckpt_path: str, device) -> nn.Module:
    state = torch.load(ckpt_path, map_location=device)
    if isinstance(state, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in state and isinstance(state[key], dict):
                state = state[key]
                break
    model.load_state_dict(state)
    model.to(device).eval()
    return model


def load_models(predictor_ckpt: str, classifier_ckpt: str, device):
    predictor = _load_state(MSAMConvLSTM(input_channels=1), predictor_ckpt, device)
    classifier = _load_state(CNNTransformer(num_classes=3, input_shape=(3, 32, 64)),
                             classifier_ckpt, device)
    return predictor, classifier


@torch.no_grad()
def cascade_predict(frames, predictor, classifier, device, future_seq: int = 5):
    frames = np.asarray(frames, dtype=np.float32)
    window = frames[-15:]
    x = torch.as_tensor(window, device=device).unsqueeze(0).unsqueeze(2)  # (1, 15, 1, H, W)
    pred = predictor(x, future_seq=future_seq)[0, :, 0].cpu().numpy()      # (future_seq, H, W)
    results = []
    for i, fr in enumerate(pred):
        small = cv2.resize(fr, (64, 32), interpolation=cv2.INTER_AREA)     # (32, 64)
        t = torch.as_tensor(small, dtype=torch.float32, device=device)
        t = t.unsqueeze(0).repeat(3, 1, 1).unsqueeze(0)                    # (1, 3, 32, 64)
        prob = torch.softmax(classifier(t), dim=1)[0].cpu().numpy()
        cls = int(prob.argmax())
        results.append({"step": i + 1, "state": CLASS_NAMES[cls], "confidence": float(prob[cls])})
    return results
